In [96]:
import sys
import os
import pickle
from pathlib import Path
from abc import ABC, abstractmethod
from scipy.fft import fft
import numpy as np
from Data.environment_classes import Drone, Radar, Context
from Data.synthetic_dataset_generator import DatasetMetadata, DataRequest
from Data.noise_models import AdditiveWhiteGaussianNoise

In [72]:
class DatasetClass(ABC):
	def __init__(self, dataset_metadata: DatasetMetadata):
		self.dataset_metadata = dataset_metadata
		pass

	@abstractmethod
	def load_one_data_object(self):
		pass

	@abstractmethod
	def load_all_data(self):
		pass


class SyntheticDataset(DatasetClass):
	def __init__(self, dataset_metadata: DatasetMetadata):
		super().__init__(dataset_metadata)
		pass

	def load_one_data_object(self):
		md = self.dataset_metadata
		full_path = md.save_path / f"{md.filename}.{md.file_format}"
		if md.file_format == "pkl":
			with open(full_path, 'rb') as f:
				obj = pickle.load(f)
			return obj
		else:
			raise ValueError("Only pkl files are supported")

	def load_all_data(self):
		md = self.dataset_metadata
		full_path = md.save_path / f"{md.filename}.{md.file_format}"
		data = []
		with open(full_path, 'rb') as f:
			try:
				while True:
					data.append(pickle.load(f))
			except EOFError:
				pass
		return data

In [73]:
PROJECT_ROOT = Path().cwd().parent
load_path = PROJECT_ROOT / "Data" / "datasets" / "time_domain" / "training_dataset.pkl"

load_path = Path(load_path)
md = DatasetMetadata.create_from_path(load_path)
synt_dataset = SyntheticDataset(dataset_metadata=md)

In [87]:
obj = synt_dataset.load_one_data_object()
obj['signal'].shape

(100,)

In [94]:
class DataParser(ABC):
	def __init__(self):
		pass

	@abstractmethod
	def parse_data(self, data):
		pass

In [111]:
class TimeDomainDataParser(ABC):
	def __init__(self):
		super().__init__()

	def parse_data(self, data, bin_size=10):
		binned_data = self.bin_data(data, bin_size)
		freq_data = self.discrete_fourier_transform(binned_data)
		modulus_data = self.compute_modulus(freq_data)
		return modulus_data

	def bin_data(self, data, bin_size=10):
		bin_array = []
		for i in range(data.shape[0]//bin_size):
			bin_array.append(np.average(data[bin_size*i:bin_size*(i+1)]))
		return np.array(bin_array)

	def discrete_fourier_transform(self, time_domain_data):
		frequency_domain_data = fft(time_domain_data)
		return frequency_domain_data

	def compute_modulus(self, data):
		modulus_data = np.absolute(data)
		return modulus_data

In [112]:
td_data_parser = TimeDomainDataParser()
obj['binned_signal'] = td_data_parser.parse_data(obj['signal'])
obj['binned_signal']

array([1431.87639266, 1483.02211242,  200.79437209, 1689.87405571,
       3131.6731831 , 1241.74208384, 1725.94837864, 2667.91095873,
       2089.65492133,  954.9438663 ])